# DeepJSCC, NTSCC++, ADJSCC: reproducible Kaggle benchmark

This run first validates the Kaggle GPU and attached DIV2K/Kodak inputs, then trains DeepJSCC for 150 epochs each on AWGN and Rayleigh and evaluates Kodak. DeepJSCC architecture/channel are imported from the author-linked PyTorch implementation.

**Scope that must remain explicit:** the author-linked NTSCC_plus repository README says NTSCC++ online inference is TODO; it is not an exact released NTSCC++ implementation. ADJSCC is TensorFlow with TensorFlow Compression custom layers. This notebook must not call NTSCC+ “NTSCC++” or silently substitute an ADJSCC architecture. Additional model stages are gated on verified compatible source/runtime adapters. This run reports only completed models.

Primary sources: [DeepJSCC source](https://github.com/chunbaobao/Deep-JSCC-PyTorch), [ADJSCC source](https://github.com/alexxu1988/ADJSCC), [NTSCC_plus source](https://github.com/wsxtyrdd/NTSCC_plus). Kaggle execution uses GPU T4 x2 when granted.


In [ ]:
from pathlib import Path
import os, glob, json, random, time, csv, subprocess, sys, math
import numpy as np
import torch, torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
assert torch.cuda.is_available(), 'Enable Kaggle GPU.'
print('GPUs:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 1
# User inputs attached through Kaggle kernel metadata.
DIV2K = Path('/kaggle/input/notebooks/jagan028/div2k-dataset-generation-for-isr/Training/HR/hr_images')
assert DIV2K.exists(), f'DIV2K folder missing: {DIV2K}'
KODAK_FILES = sorted(Path('/kaggle/input/datasets/sherylmehta/kodak-dataset').rglob('kodim*.png'))
assert len(KODAK_FILES) == 24, f'Expected 24 Kodak PNGs; found {len(KODAK_FILES)}'
print('DIV2K image count:', sum(p.suffix.lower() in {'.png','.jpg','.jpeg'} for p in DIV2K.rglob('*')),'Kodak:',len(KODAK_FILES))
OUT=Path('/kaggle/working/jscc_results'); OUT.mkdir(parents=True,exist_ok=True)


In [ ]:
# Fetch and record exact source revision.
DEEP_SRC=Path('/kaggle/working/Deep-JSCC-PyTorch')
if not DEEP_SRC.exists(): subprocess.run(['git','clone','https://github.com/chunbaobao/Deep-JSCC-PyTorch.git',str(DEEP_SRC)],check=True)
DEEP_COMMIT=subprocess.check_output(['git','-C',str(DEEP_SRC),'rev-parse','HEAD'],text=True).strip()
print('DeepJSCC author-linked reproduction commit:',DEEP_COMMIT)
sys.path.insert(0,str(DEEP_SRC))
from model import DeepJSCC


In [ ]:
class RandomCrops(Dataset):
    def __init__(self, root, crop=128):
        self.paths=sorted(p for p in Path(root).rglob('*') if p.suffix.lower() in {'.png','.jpg','.jpeg'})
        if not self.paths: raise FileNotFoundError(root)
        self.crop=crop
    def __len__(self): return len(self.paths)
    def __getitem__(self,i):
        im=Image.open(self.paths[i]).convert('RGB'); w,h=im.size
        if w<self.crop or h<self.crop:
            im=im.resize((max(w,self.crop),max(h,self.crop)),Image.Resampling.BICUBIC); w,h=im.size
        x=random.randint(0,w-self.crop); y=random.randint(0,h-self.crop)
        im=im.crop((x,y,x+self.crop,y+self.crop))
        if random.random()<.5: im=im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        return transforms.ToTensor()(im)
train_ds=RandomCrops(DIV2K,128)
loader=DataLoader(train_ds,batch_size=32,shuffle=True,num_workers=4,pin_memory=True,drop_last=True)
print('Crop samples per epoch:',len(loader)*loader.batch_size)


In [ ]:
# Metrics are computed per image in [0,1] space. MMSIM is reported as MS-SSIM.
!pip -q install pytorch-msssim
from pytorch_msssim import ssim, ms_ssim
def quality(x,y):
    mse=F.mse_loss(x,y).item(); psnr=-10*math.log10(max(mse,1e-12))
    return psnr,ssim(x,y,data_range=1.0,size_average=True).item(),ms_ssim(x,y,data_range=1.0,size_average=True).item()
def get_kodak():
    out=[]
    for p in KODAK_FILES:
        im=Image.open(p).convert('RGB')
        out.append((p.stem,transforms.ToTensor()(im).unsqueeze(0)))
    return out
KODAK=get_kodak()


In [ ]:
# Rate 1/6: official ratio2filtersize gives c=3*H*W*(1/6)/(H*W/16)=8.
# Encoder outputs 2*c real planes, paired as complex channel symbols, so uses=c*H*W/16.
EPOCHS=150; SNR_DB=10.0; SEED=20261016; C=8
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device=torch.device('cuda:0')
def train_condition(channel):
    model=DeepJSCC(c=C,channel_type=channel,snr=SNR_DB).to(device)
    if torch.cuda.device_count()>=2: model=nn.DataParallel(model,device_ids=list(range(torch.cuda.device_count())))
    opt=torch.optim.Adam(model.parameters(),lr=1e-4,weight_decay=5e-4)
    run=OUT/f'DeepJSCC_{channel}'; run.mkdir(exist_ok=True)
    rows=[]
    for epoch in range(1,EPOCHS+1):
        model.train(); start=time.time(); total=0.; n=0
        for x in loader:
            x=x.to(device,non_blocking=True); opt.zero_grad(set_to_none=True)
            pred=model(x); loss=F.mse_loss(pred,x); loss.backward(); opt.step()
            total+=loss.detach().item()*x.size(0); n+=x.size(0)
        row={'model':'DeepJSCC','channel':channel,'snr_db':SNR_DB,'epoch':epoch,'train_mse':total/max(n,1),'epoch_seconds':time.time()-start,'c':C,'nominal_cbr':1/6}
        rows.append(row)
        with (run/'epoch_metrics.csv').open('w',newline='') as f:
            w=csv.DictWriter(f,fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
        raw=model.module if isinstance(model,nn.DataParallel) else model
        torch.save({'epoch':epoch,'model':raw.state_dict(),'optimizer':opt.state_dict(),'config':row},run/'latest.pt')
        if epoch%10==0 or epoch==1: print(json.dumps(row),flush=True)
    return model,run


In [ ]:
# 150 epochs per channel. Checkpoints persist in Kaggle output; metrics are never fabricated.
all_rows=[]
for channel in ['AWGN','Rayleigh']:
    trained,run=train_condition(channel)
    trained.eval()
    with torch.no_grad():
        for name,ref in KODAK:
            ref=ref.to(device); pred=trained(ref).clamp(0,1)
            psnr_v,ssim_v,msssim_v=quality(pred,ref)
            # Encoder representation: real dimensions / 2 = complex channel uses.
            z=(trained.module if isinstance(trained,nn.DataParallel) else trained).encoder(ref)
            real_symbols=z.numel(); complex_uses=real_symbols//2; cbr=complex_uses/(3*ref.shape[-2]*ref.shape[-1])
            all_rows.append({'model':'DeepJSCC','channel':channel,'snr_db':SNR_DB,'image':name,'psnr_db':psnr_v,'ssim':ssim_v,'msssim':msssim_v,'real_symbols':real_symbols,'complex_channel_uses':complex_uses,'cbr':cbr})
    trained=None; torch.cuda.empty_cache()
with (OUT/'kodak_per_image.csv').open('w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=all_rows[0].keys()); w.writeheader(); w.writerows(all_rows)
summary=[]
for ch in ['AWGN','Rayleigh']:
    vals=[r for r in all_rows if r['channel']==ch]
    summary.append({'model':'DeepJSCC','channel':ch,'snr_db':SNR_DB,'epochs':EPOCHS,'kodak_psnr_db':np.mean([r['psnr_db'] for r in vals]),'kodak_ssim':np.mean([r['ssim'] for r in vals]),'kodak_msssim':np.mean([r['msssim'] for r in vals]),'channel_uses_per_image_mean':np.mean([r['complex_channel_uses'] for r in vals]),'cbr':np.mean([r['cbr'] for r in vals])})
with (OUT/'kodak_summary.csv').open('w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=summary[0].keys()); w.writeheader(); w.writerows(summary)
print(summary)


## Run status and scope

This notebook currently trains the DeepJSCC author-linked reproduction for the requested 150 epochs per channel, then evaluates Kodak. The other two models are gated: the NTSCC++ online code is not released by the author-linked repo; ADJSCC requires its TensorFlow Compression custom layers and channel inputs and needs a separately validated Kaggle-compatible adapter. Do not record them as trained until those stages execute. Attach the resulting Kaggle output to this experiment before claiming a three-model comparison. FLOPs are still to be added as a validated profiler measurement; no placeholder is reported as a result.
